# Breast-cancer real-data study

Run the same predefined experiment on the patient-level expression matrix.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from stablewide.notebook_utils import find_project_root, run_experiment, read_outputs, core_subsample_table
ROOT = find_project_root(ROOT)
print(ROOT)

In [ ]:
from stablewide.omics import validate_matrix_and_labels

X_CSV = ROOT / "data" / "processed" / "brca_mrna.csv"
Y_CSV = ROOT / "data" / "processed" / "brca_labels.csv"
if not X_CSV.exists() or not Y_CSV.exists():
    raise FileNotFoundError("Prepare brca_mrna.csv and brca_labels.csv in notebook 04 first.")

X_real, y_real, LABEL_COL = validate_matrix_and_labels(X_CSV, Y_CSV, "subtype")
n = len(X_real)
n_classes = y_real[LABEL_COL].nunique()
pool_est = max(n_classes + 1, int(round(n * 0.8)))
raw_sizes = [int(pool_est * f) for f in (0.75, 0.55, 0.40)]
sizes = sorted({max(n_classes, s) for s in raw_sizes if n_classes <= s < pool_est}, reverse=True)
if not sizes:
    raise ValueError(f"Cohort too small for a meaningful subsampling study: n={n}, classes={n_classes}")
SIZES = ",".join(map(str, sizes))
print(f"patients={n}, classes={n_classes}, approximate development pool={pool_est}, subsample sizes={SIZES}")

In [ ]:
OUT = ROOT / "outputs" / "05_brca_real"
args = [
    "--x", str(X_CSV),
    "--y", str(Y_CSV),
    "--label_col", LABEL_COL,
    "--methods", "anova,rf,l1,l2,attention",
    "--outer_splits", "5",
    "--n_reps", "20",
    "--n_seeds", "10",
    "--sizes", SIZES,
    "--n_features", "2000",
    "--topk", "5,10,20,50",
    "--decision_k", "10",
    "--known_genes", "ESR1,FOXA1,GATA3,FOXC1,CCDC170",
    "--run_tuned_performance",
    "--save_feature_tables",
    "--out", str(OUT),
]
print("Ready to run:", OUT)

In [ ]:
run_experiment(ROOT, args)

In [ ]:
res = read_outputs(OUT)
print(res.get("decision", "No decision file"))
core_subsample_table(res["aggregate_results.csv"], k=10).round(3)

The subsample sizes above are derived from the actual cohort size. For a tiny cohort, the correct conclusion may simply be that biomarker ranking is not supportable at that sample size.